# 01_02 Neural networks

*📖 Book: appendix A*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **the basics** (chapter 1): the ideas every stage uses
>
> Project files this connects to:
>
> - `src/homer_ai/model.py`
>
> **This notebook:** The model in `model.py` is a neural network. What is that, and what are its 11,840 weights?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- What a neural network is: layers of weights, numbers in → numbers out
- How one layer turns 3 numbers into 2
- How to count a network's weights
- How our model's layers stack like floors

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **neural network** | layers of weights that turn numbers in into numbers out | our `HomerGreekLLM` |
| **layer** | one step of the calculation: numbers go in, are mixed using weights, and new numbers come out | one floor of a building |
| **linear layer** | the simplest layer: each output is the inputs multiplied by weights and added up, plus one extra number | `nn.Linear(3, 2)` |
| **bias** | the extra number a linear layer adds to each output | one per output |

## 3. The idea

### ❓ What is a neural network?

A calculation made of **layers**. Numbers go in at the bottom, each layer mixes them using its weights (its adjustable numbers), and numbers come out at the top. That's all it is, really.

The name comes from a loose comparison with brain cells (neurons), but you don't need any biology to use one.

### ❓ Is there a picture?

Yes. Think of an office building. The numbers go up the stairs floor by floor, and on every floor the workers adjust knobs (the weights). There's a drawing of it, [the learning building](../../docs/neural-network-building.html), in this repo's `docs/` folder.

### ❓ Where do the weights come from?

They start out **random**, so a brand-new network gives random answers. Training (notebook 01_03) is what adjusts the weights until the answers are good.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ How do I make one layer?

PyTorch has one ready-made, the **linear layer**. `nn.Linear(3, 2)` takes 3 numbers in and gives 2 numbers out. Each output is the 3 inputs multiplied by weights and added up, plus one extra number, the bias. `torch` is PyTorch, and `torch.tensor` holds the numbers (see [A_02](../A_toolbox/A_02_pytorch_tensors.ipynb)).

```
[1, 2, 3]   →   layer: 6 weights + 2 biases   →   [?, ?]
 3 numbers in                                  2 numbers out
```

👉 **Predict first:** we feed in `[1, 2, 3]`. Will the 2 outputs mean anything?

In [ ]:
import torch
from torch import nn

torch.manual_seed(42)  # the same "random" weights every run
layer = nn.Linear(3, 2)

x = torch.tensor([1.0, 2.0, 3.0])
print(layer(x))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Two numbers, `0.7130` and `0.9659`, and they mean nothing yet, because the weights are random. (You can ignore `grad_fn=…`. That's PyTorch noting how the numbers were made, which it needs for learning later.)

</details>

### ❓ How many weights does that layer have?

Each of the 2 outputs has one weight per input (so 3), plus its bias. That's 2 × 3 + 2 = 8. `.parameters()` lists all the adjustable numbers, and `.numel()` counts the numbers in each.

👉 **Predict first:** 8?

In [ ]:
print(layer.weight.shape, layer.bias.shape)
print(sum(p.numel() for p in layer.parameters()))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

A 2 × 3 table of weights, 2 biases, and 8 adjustable numbers in total. (PyTorch calls every adjustable number a **parameter**. In this course we just say "the weights" for weights and biases together.)

</details>

**Look at:** the 3 dots on the left are the numbers going in, and the 2 on the right are the numbers coming out. Each of the 6 lines is one weight, and each green dot is a bias.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import (
    ACCENT,
    GOOD,
    INK,
    MUTED,
    diagram_axes,
    dot,
    studio_figure,
    tag,
)

with torch.no_grad():
    outputs = layer(x).tolist()
fig = studio_figure("One layer", "3 numbers in, 2 numbers out", (8, 6))
ax = diagram_axes(fig)
inputs = [(2.0, 4.6), (2.0, 3.2), (2.0, 1.8)]
outs = [(7.0, 4.0), (7.0, 2.4)]
for (x0, y0) in inputs:
    for (x1, y1) in outs:
        ax.plot([x0, x1], [y0, y1], color=MUTED, linewidth=1.4, zorder=1)
for (px, py), value in zip(inputs, [1, 2, 3], strict=True):
    dot(ax, px, py, str(value), color=MUTED)
for (px, py), value in zip(outs, outputs, strict=True):
    dot(ax, px, py, f"{value:.2f}", r=0.5, color=ACCENT, fontsize=10)
    ax.plot([px + 0.5, px + 1.2], [py, py], color=GOOD, linewidth=1.4)
    dot(ax, px + 1.5, py, "b", r=0.3, color=GOOD, fontsize=10)
tag(ax, 2.0, 5.6, "3 numbers in", INK, 11, weight="bold")
tag(ax, 7.0, 5.1, "2 numbers out", INK, 11, weight="bold")
tag(ax, 4.5, 0.9, "6 lines = 6 weights", MUTED, 10)
tag(ax, 8.2, 3.2, "2 bias dots (b)\n= 2 biases", GOOD, 9)
tag(ax, 4.5, 0.4, "6 weights + 2 biases = 8", ACCENT, 11, weight="bold")
plt.show()

### ❓ What happens if I stack two layers?

The output of the first floor becomes the input of the second. `nn.Sequential` stacks layers in order.

👉 **Predict first:** a 3 → 4 layer, then a 4 → 2 layer. How many weights in total?

In [ ]:
torch.manual_seed(42)
net = nn.Sequential(nn.Linear(3, 4), nn.Linear(4, 2))

print(net(x))
print(sum(p.numel() for p in net.parameters()))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Still 2 numbers out, but now **26** weights: (4 × 3 + 4) + (2 × 4 + 2) = 16 + 10.

</details>

## 5. Full example

### ❓ What are the floors of our own model?

`HomerGreekLLM` in `src/homer_ai/model.py` is a neural network too. Here are its floors, bottom to top:

```
head              → a score for each of the 137 possible next characters
query, key, value → the attention floor: which earlier characters matter (chapter 3)
embedding         → turns each character's ID into 32 numbers (chapter 2)
```

We open up each floor in the chapters that follow. Here, let's just count the weights.

👉 **Predict first:** more or fewer than 10,000?

In [ ]:
from homer_ai.model import HomerGreekLLM

model = HomerGreekLLM(vocab_size=137, embed_dim=32)

for name, p in model.named_parameters():
    print(f"{name:20} {str(tuple(p.shape)):10} {p.numel():6}")
print("total:", sum(p.numel() for p in model.parameters()))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

So that's 4,384 in the embedding (137 × 32), 1,024 each in query, key and value (32 × 32), and another 4,384 in the head (137 × 32). **11,840** in total.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Why are the starting weights random, and not zero?

If every weight started out the same, every worker on a floor would do the same job, and training could never make them different. A random start gives each one its own direction to grow in.

## 6. Recap

- A neural network is layers of weights: numbers in, numbers out.
- A linear layer multiplies inputs by weights, adds them up, and adds a bias.
- Our model stacks embedding → attention → head: 11,840 weights.

**You can now:**

- make a layer with `nn.Linear` and say how many weights it has
- stack layers, and count a whole model's weights

**Next:** [01_03: How a program learns](01_03_how_a_program_learns.ipynb).